In [0]:
# ==========================================
# SILVER LAYER NOTEBOOK (FULL CODE)
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# 1. Create Silver Volume (Run Once)
# ------------------------------------------

spark.sql("""
CREATE VOLUME IF NOT EXISTS retail_catalog1.silver.raw_volume
""")

# ------------------------------------------
# 2. Read Bronze Delta Table
# ------------------------------------------

bronze_df = (
    spark.readStream
    .format("delta")
    .table("retail_catalog1.bronze.transactions")
)

# ------------------------------------------
# 3. Clean Data
# ------------------------------------------

cleaned_df = (
    bronze_df

    # Convert amount safely
    .withColumn(
        "amount",
        F.when(
            F.col("amount").rlike("^-?[0-9]+(\\.[0-9]+)?$"),
            F.col("amount").cast("double")
        ).otherwise(None)
    )

    # Handle invalid timestamps safely
    .withColumn(
        "timestamp",
        F.when(
            F.col("timestamp").rlike("^\\d{4}-\\d{2}-\\d{2}"),
            F.to_timestamp("timestamp")
        ).otherwise(None)
    )

    # Remove bad records
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("amount").isNotNull())
    .filter(F.col("timestamp").isNotNull())

    # Remove negative amounts
    .filter(F.col("amount") > 0)

    # Remove duplicates
    .dropDuplicates(["transaction_id"])

    # Select required columns only
    .select(
        "transaction_id",
        "customer_id",
        "amount",
        "store_id",
        "timestamp",
        "payment_type"
    )
)

# ------------------------------------------
# 4. Create Silver Table If Not Exists
# ------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS retail_catalog1.silver.transactions (
    transaction_id STRING,
    customer_id STRING,
    amount DOUBLE,
    store_id STRING,
    timestamp TIMESTAMP,
    payment_type STRING
)
USING DELTA
""")

# ------------------------------------------
# 5. Write Stream To Silver
# ------------------------------------------

query = (
    cleaned_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/retail_catalog1/silver/raw_volume/_checkpoints/transactions"
    )
    .trigger(availableNow=True)
    .toTable("retail_catalog1.silver.transactions")
)

query.awaitTermination()

# ------------------------------------------
# 6. Validation
# ------------------------------------------

display(
    spark.sql("""
    SELECT *
    FROM retail_catalog1.silver.transactions
    ORDER BY timestamp DESC
    """)
)

transaction_id,customer_id,amount,store_id,timestamp,payment_type,_rescued_data
T1021,C121,750.0,S01,INVALID_DATE,Cash,null
T1030,C130,620.0,S02,2025-06-02 09:25:00,Cash,null
T1030,C130,620.0,S02,2025-06-02 09:25:00,Cash,null
T1029,C129,1100.0,S01,2025-06-02 09:20:00,UPI,null
T1029,C129,1100.0,S01,2025-06-02 09:20:00,UPI,null
T1028,C128,950.0,S03,2025-06-02 09:15:00,Credit,null
T1028,C128,950.0,S03,2025-06-02 09:15:00,Credit,null
T1027,C127,800.0,S04,2025-06-02 09:10:00,Debit,null
T1027,C127,800.0,S04,2025-06-02 09:10:00,Debit,null
T1026,C126,420.0,S01,2025-06-02 09:05:00,Cash,null
